In [1]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import joblib
import shap
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from xgboost import XGBClassifier

# from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import make_scorer, precision_score, recall_score, classification_report
from imblearn.over_sampling import SMOTE
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical

from sklearn.model_selection import LearningCurveDisplay, learning_curve
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import f1_score, make_scorer

from scipy import stats

import math
from sklearn.cluster import AgglomerativeClustering

import requests
import logging



# CSF EDITS
import anndata as ad
import scanpy as sc
import pandas as pd
import seaborn as sns

gpu = True
if gpu:
    #GPU Libraries
    import cudf
    import cupy as cp



# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('bcell_analysis.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Log the start of the analysis
logger.info('Starting CD4 AUTO NO NAIVE PBMC solo 2 dataset analysis')    

2025-06-15 15:22:43,554 - __main__ - INFO - Starting CD4 AUTO NO NAIVE PBMC solo 2 dataset analysis


In [2]:
os.getcwd()
dataset_dir = "PBMC_BCELLS_2DATASET"

logger.info('Loading dataset...')
# Loading dataset 
dataset = sc.read_h5ad(os.path.join("..", "Dataset","MatriciAddestramento", dataset_dir+".h5ad"))

Store_dir = os.path.join(os.getcwd(), "Store", dataset_dir)

if not os.path.exists(Store_dir):
    os.makedirs(Store_dir)
logger.info('Store directory created...')

os.chdir(Store_dir)
logger.info('Chdir directory changed...')

2025-06-15 15:22:43,561 - __main__ - INFO - Loading dataset...
2025-06-15 15:22:45,255 - __main__ - INFO - Store directory created...
2025-06-15 15:22:45,256 - __main__ - INFO - Chdir directory changed...


In [3]:
print(dataset)

AnnData object with n_obs × n_vars = 3505 × 19061
    obs: 'condition', 'sample', 'tissue', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'predicted_labels', 'over_clustering', 'majority_voting', 'dataset', 'generalized_celltype', '_scvi_batch', '_scvi_labels', 'concat_batch'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'dataset_colors', 'generalized_celltype_colors', 'majority_voting_colors', 'neighbors', 'pca', 'sample_colors', 'umap'
    obsm: 'X_pca', 'X_umap', 'corrected_latent', 'latent'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'


In [5]:
print(list(dataset.var_names))

['OR4F5', 'AP006222.2', 'FAM87B', 'LINC00115', 'FAM41C', 'SAMD11', 'NOC2L', 'KLHL17', 'PLEKHN1', 'PERM1', 'HES4', 'ISG15', 'AGRN', 'RNF223', 'C1orf159', 'LINC01342', 'TTLL10-AS1', 'TTLL10', 'TNFRSF18', 'TNFRSF4', 'SDF4', 'B3GALT6', 'UBE2J2', 'SCNN1D', 'ACAP3', 'PUSL1', 'CPTP', 'TAS1R3', 'DVL1', 'MXRA8', 'AURKAIP1', 'CCNL2', 'MRPL20', 'ANKRD65', 'VWA1', 'ATAD3C', 'ATAD3B', 'ATAD3A', 'TMEM240', 'SSU72', 'AL645728.1', 'MIB2', 'MMP23B', 'CDK11B', 'SLC35E2B', 'CDK11A', 'NADK', 'GNB1', 'CALML6', 'TMEM52', 'CFAP74', 'GABRD', 'PRKCZ', 'FAAP20', 'SKI', 'MORN1', 'RER1', 'PEX10', 'PLCH2', 'PANK4', 'HES5', 'TNFRSF14', 'MMEL1', 'TTC34', 'PRDM16', 'ARHGEF16', 'MEGF6', 'TPRG1L', 'WRAP73', 'TP73', 'CCDC27', 'SMIM1', 'LRRC47', 'CEP104', 'DFFB', 'C1orf174', 'LINC01134', 'LINC01346', 'AJAP1', 'NPHP4', 'KCNAB2', 'CHD5', 'RPL22', 'RNF207', 'ICMT', 'LINC00337', 'GPR153', 'ACOT7', 'HES2', 'ESPN', 'TNFRSF25', 'PLEKHG5', 'NOL9', 'TAS1R1', 'ZBTB48', 'KLHL21', 'PHF13', 'THAP3', 'DNAJC11', 'CAMTA1', 'VAMP3', 'PER

# PREPROCESSING CLUSTERING

In [4]:
current = dataset.copy()
if gpu: 
    logger.info('GPU is enabled...')
    data = current.X
    cudf_df = cudf.DataFrame(data.astype(cp.float32), columns=current.var_names)
    logger.info('Calculating correlation matrix...')
    dataCorr = cudf_df.corr()
    logger.info('Correlation matrix calculated...')
    joblib.dump(dataCorr, "dataCorr.pkl", compress=3)
    logger.info('Correlation matrix saved...')

    #CLEAN UP GPU
    del cudf_df
    cp.get_default_memory_pool().free_all_blocks()
else:
    logger.info('GPU is disabled...')
    data = current.X
    logger.info('Calculating correlation matrix...')
    dataCorr = pd.DataFrame(data, columns=current.var_names).corr()
    logger.info('Correlation matrix calculated...')
    joblib.dump(dataCorr, "dataCorr.pkl", compress=3)
    logger.info('Correlation matrix saved...')

2025-05-24 16:26:35,380 - __main__ - INFO - GPU is enabled...
2025-05-24 16:26:39,718 - __main__ - INFO - Calculating correlation matrix...
2025-05-24 16:26:46,212 - __main__ - INFO - Correlation matrix calculated...
2025-05-24 16:28:58,437 - __main__ - INFO - Correlation matrix saved...


In [5]:
logger.info('Loading correlation matrix...')
dataCorr = joblib.load( "dataCorr.pkl") 
logger.info('Correlation matrix loaded...')
dataCorr = dataCorr.to_pandas()
logger.info('Correlation matrix converted to pandas...')
clusters = []
# Get absolute correlation values once
abs_corr = np.abs(dataCorr)
logger.info('Absolute correlation matrix calculated...')
for gene_i in dataCorr.index:
    # Get all correlations >= 0.9 for this gene
    high_corr_mask = abs_corr.loc[gene_i] >= 0.9
    cluster = dataCorr.columns[high_corr_mask].tolist()
    
    if len(cluster) > 1:  # Only add if cluster has more than 1 gene
        clusters.append(cluster)
logger.info('Clusters calculated...')
logger.info(f'Width of clusters:  {len(clusters)}')
joblib.dump(clusters, "clusters.pkl")
logger.info('Clusters saved...')

2025-05-24 16:29:30,670 - __main__ - INFO - Loading correlation matrix...
2025-05-24 16:29:57,982 - __main__ - INFO - Correlation matrix loaded...
2025-05-24 16:30:17,899 - __main__ - INFO - Correlation matrix converted to pandas...
2025-05-24 16:30:18,288 - __main__ - INFO - Absolute correlation matrix calculated...
2025-05-24 16:30:29,197 - __main__ - INFO - Clusters calculated...
2025-05-24 16:30:29,198 - __main__ - INFO - Width of clusters:  3973
2025-05-24 16:30:29,542 - __main__ - INFO - Clusters saved...


In [6]:
logger.info('Loading clusters...')
clusters = joblib.load( "clusters.pkl")
logger.info('Clusters loaded...')
gene_list = list(dataset.var_names)
logger.info(f'Length of gene list:  {len(gene_list)}')

for cluster in clusters:
    for gene in cluster:
        if gene in gene_list:
            gene_list.remove(gene)
logger.info('Genes removed from gene list...')
logger.info(f'Length of gene list: {len(gene_list)}')

2025-05-24 16:31:16,404 - __main__ - INFO - Loading clusters...
2025-05-24 16:31:16,594 - __main__ - INFO - Clusters loaded...
2025-05-24 16:31:16,596 - __main__ - INFO - Length of gene list:  19061
2025-05-24 16:31:57,531 - __main__ - INFO - Genes removed from gene list...
2025-05-24 16:31:57,532 - __main__ - INFO - Length of gene list: 15088


In [7]:
current = dataset.copy()
data = pd.DataFrame(current.X, columns=current.var_names)
representative = {}
logger.info('Calculating variances...')
# Calculate variances once for all variables
all_variances = {var: data[var].var() for var in data.columns}
logger.info('Variances calculated...')
# Sort all variables by variance (highest to lowest) once
sorted_vars = sorted(all_variances.items(), key=lambda item: item[1], reverse=True)
sorted_vars_list = [var for var, _ in sorted_vars]
logger.info('Variances sorted...')
# Process clusters by size (largest first)
logger.info('Processing clusters...')
for cluster in sorted(clusters, key=len, reverse=True):
    cluster_tuple = tuple(cluster)
    
    # Find the highest variance variable in this cluster that isn't already used
    found_representative = False
    for var in sorted_vars_list:
        if var in cluster and var not in representative.values():
            representative[cluster_tuple] = var
            found_representative = True
            break
    
    # If we couldn't find an unused representative, use the highest variance one anyway
    if not found_representative:
        for var in sorted_vars_list:
            if var in cluster:
                representative[cluster_tuple] = var
                break
logger.info('Representative calculated...')
# Filter out subsets more efficiently
unique_clusters = {}
cluster_tuples = list(representative.keys())
logger.info('Cluster tuples sorted...') 
# Sort by length once
cluster_tuples.sort(key=len)
logger.info('Cluster tuples sorted...')     
# Use a more efficient algorithm to filter subsets
logger.info('Filtering subsets...')
for i, cluster in enumerate(cluster_tuples):
    is_unique = True
    cluster_set = set(cluster)
    
    # Only check against larger clusters
    for j in range(i+1, len(cluster_tuples)):
        larger_cluster = cluster_tuples[j]
        if cluster_set.issubset(set(larger_cluster)):
            is_unique = False
            break
    
    if is_unique:
        unique_clusters[cluster] = representative[cluster]
logger.info('Unique clusters calculated...')
joblib.dump(unique_clusters, "uniqueClusters.pkl")
logger.info('Unique clusters saved...')

2025-05-24 16:31:57,583 - __main__ - INFO - Calculating variances...
2025-05-24 16:32:01,922 - __main__ - INFO - Variances calculated...
2025-05-24 16:32:01,937 - __main__ - INFO - Variances sorted...
2025-05-24 16:32:01,938 - __main__ - INFO - Processing clusters...
2025-05-24 16:32:10,252 - __main__ - INFO - Representative calculated...
2025-05-24 16:32:10,253 - __main__ - INFO - Cluster tuples sorted...
2025-05-24 16:32:10,254 - __main__ - INFO - Cluster tuples sorted...
2025-05-24 16:32:10,255 - __main__ - INFO - Filtering subsets...
2025-05-24 16:32:25,983 - __main__ - INFO - Unique clusters calculated...
2025-05-24 16:32:26,279 - __main__ - INFO - Unique clusters saved...


In [8]:

def find_key_by_value_in_tuple(
    unique_clusters_path, target_value: str
) -> tuple | None:
    """
    Loads a dictionary of unique clusters from a file and searches for a key
    (cluster tuple) that has the specified target value as its value.

    Args:
        unique_clusters_path (str): The path to the file containing the
            unique clusters dictionary (created using joblib.dump).
        target_value (str): The value to search for among the values in the
            dictionary.

    Returns:
        tuple | None: The key (cluster tuple) that has the target value as its
            value. Returns None if the target value is not found.
    """

    for cluster_tuple, value in unique_clusters.items():
        if value == target_value:
            return cluster_tuple

    return None
print(find_key_by_value_in_tuple(unique_clusters, "HLA-DRB1"))
print("HLA-DRB1" in gene_list)

('SNX2', 'CD74', 'HLA-DRA', 'HLA-DRB1', 'HLA-DQA1', 'HLA-DQB1', 'HLA-DMB', 'HLA-DMA', 'HLA-DPA1', 'HLA-DPB1', 'CD37')
False


In [9]:
gene_list.extend(gene for gene in unique_clusters.values() if gene not in gene_list) 
logger.info(f'Length of gene list: {len(gene_list)}')
logger.info(f'Adding back HLA-DRB1 cluster...')
cluster = ['SNX2', 'CD74', 'HLA-DRA', 'HLA-DRB1', 'HLA-DQA1', 'HLA-DQB1', 'HLA-DMB', 'HLA-DMA', 'HLA-DPA1', 'HLA-DPB1', 'CD37']
for gene in cluster:
    if gene not in gene_list:
        gene_list.append(gene)  # Append each gene individually

2025-05-24 16:32:52,272 - __main__ - INFO - Length of gene list: 17402
2025-05-24 16:32:52,273 - __main__ - INFO - Adding back HLA-DRB1 cluster...


In [10]:
logger.info('Declustering dataset...')
datasetDeclustered = dataset[:, gene_list]
logger.info('Dataset declustered...')
datasetDeclustered.write_h5ad("datasetDeclustered.h5ad")#Dataset che viene utilizzato per tutte le analisi successive
logger.info('Dataset declustered saved...')
print(datasetDeclustered)

2025-05-24 16:32:56,706 - __main__ - INFO - Declustering dataset...
2025-05-24 16:32:56,723 - __main__ - INFO - Dataset declustered...
2025-05-24 16:32:57,708 - __main__ - INFO - Dataset declustered saved...


View of AnnData object with n_obs × n_vars = 3505 × 17410
    obs: 'condition', 'sample', 'tissue', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'predicted_labels', 'over_clustering', 'majority_voting', 'dataset', 'generalized_celltype', '_scvi_batch', '_scvi_labels', 'concat_batch'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'dataset_colors', 'generalized_celltype_colors', 'majority_voting_colors', 'neighbors', 'pca', 'sample_colors', 'umap'
    obsm: 'X_pca', 'X_umap', 'corrected_latent', 'latent'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'


# DIVISIONE TRAIN VALIDATION E TEST DEL DATASET

In [4]:
from pprint import pprint
import math
sample_len = len(dataset.obs['sample'].unique().tolist())

# proportion for test and validation
# 80 20 20 
logger.info('Proportion for test and validation...')
logger.info('Length of sample list: ' + str(math.floor(sample_len)))
logger.info('Training sample needed: ' + str(math.floor(sample_len * 0.6)))
logger.info('Validation sample needed: ' + str(math.floor(sample_len * 0.2)))
logger.info('Test sample needed: ' + str(math.floor(sample_len * 0.2)))

2025-06-15 15:22:47,394 - __main__ - INFO - Proportion for test and validation...
2025-06-15 15:22:47,395 - __main__ - INFO - Length of sample list: 15
2025-06-15 15:22:47,395 - __main__ - INFO - Training sample needed: 9
2025-06-15 15:22:47,396 - __main__ - INFO - Validation sample needed: 3
2025-06-15 15:22:47,397 - __main__ - INFO - Test sample needed: 3


In [5]:
import numpy as np
import pandas as pd
import math


# --- Configuration ---
dataset = sc.read_h5ad("datasetDeclustered.h5ad")
rng_seed = 46  # <--- CHANGE THIS SEED TO GET DIFFERENT SPLITS
train_proportion = 0.6
validation_proportion = 0.2
test_proportion = 0.2
# --- End Configuration ---

# 1. Get unique patient IDs and shuffle them
unique_sample_ids = dataset.obs['sample'].unique().tolist()
np.random.RandomState(rng_seed).shuffle(unique_sample_ids) # Shuffle in place

total_unique_samples = len(unique_sample_ids)

# 2. Calculate the number of samples for each set
n_train_samples = math.floor(total_unique_samples * train_proportion)
n_validation_samples = math.floor(total_unique_samples * validation_proportion)
# n_test_samples is the remainder to ensure all samples are used
# and to handle potential rounding issues if sum of proportions is not exactly 1
# or if total_unique_samples * proportion results in fractions.
n_test_samples = total_unique_samples - n_train_samples - n_validation_samples


print(f"--- Sample Allocation (based on seed: {rng_seed}) ---")
print(f"Total unique samples: {total_unique_samples}")
print(f"Targeting {train_proportion*100:.0f}% for Training: {n_train_samples} samples")
print(f"Targeting {validation_proportion*100:.0f}% for Validation: {n_validation_samples} samples")
print(f"Targeting {test_proportion*100:.0f}% (or remainder) for Test: {n_test_samples} samples")

# Adjust if rounding caused issues and we don't have enough for test
# This scenario is less likely with floor for train and val, and test as remainder,
# but good to be aware of if proportions were handled differently.
if n_train_samples + n_validation_samples + n_test_samples > total_unique_samples:
    print("Warning: Sum of calculated samples exceeds total. Adjusting test samples.")
    n_test_samples = total_unique_samples - n_train_samples - n_validation_samples
elif n_train_samples + n_validation_samples + n_test_samples < total_unique_samples:
    # If there's a shortfall (e.g. due to multiple floor operations and proportions not summing to 1)
    # assign the remainder to the training set, or distribute as preferred.
    # Here, we'll add to test to ensure it gets its intended proportion as much as possible.
    print(f"Note: {total_unique_samples - (n_train_samples + n_validation_samples + n_test_samples)} samples unallocated due to rounding. Adding to test set.")
    n_test_samples = total_unique_samples - n_train_samples - n_validation_samples


# 3. Select sample IDs for each set from the shuffled list
# Ensure that we don't try to select more samples than available if lists are small
train_sample_ids = unique_sample_ids[:n_train_samples]
validation_sample_ids = unique_sample_ids[n_train_samples : n_train_samples + n_validation_samples]
test_sample_ids = unique_sample_ids[n_train_samples + n_validation_samples:] # Takes all remaining

train_sample_ids = ['71658', '95809', 'JSB_PBMC', '49131', '74594', 'JYJ_PBMC', 'KYO_PBMC', '32190', '60249']
validation_sample_ids = ['85037', 'YYW_PBMC', '83775']
test_sample_ids = ['41540', '19270', 'KJS_PBMC']



# Recalculate actual numbers based on selection (important if total_unique_samples was small)
actual_n_train = len(train_sample_ids)
actual_n_val = len(validation_sample_ids)
actual_n_test = len(test_sample_ids)

print(f"\nActual samples selected for Training: {actual_n_train}")
print(f"Actual samples selected for Validation: {actual_n_val}")
print(f"Actual samples selected for Test: {actual_n_test}")
if actual_n_train + actual_n_val + actual_n_test != total_unique_samples:
    print(f"Warning: Mismatch in total samples after selection! Check logic. Sum: {actual_n_train + actual_n_val + actual_n_test}")


print(f"\nSelected samples for TRAINING set: {train_sample_ids[:5]}... (first 5)")
print(f"Selected samples for VALIDATION set: {validation_sample_ids[:5]}... (first 5)")
print(f"Selected samples for TEST set: {test_sample_ids[:5]}... (first 5)")


# 4. Create boolean masks for each set
train_mask = dataset.obs['sample'].isin(train_sample_ids)
validation_mask = dataset.obs['sample'].isin(validation_sample_ids)
test_mask = dataset.obs['sample'].isin(test_sample_ids)

# 5. Create AnnData subsets
train_adata = dataset[train_mask].copy()
validation_adata = dataset[validation_mask].copy()
test_adata = dataset[test_mask].copy()

# 6. Shuffle the observations within each AnnData dataset (maintaining reproducibility)
# Using the same rng_seed for this shuffling step for consistency,
# or you can use a different one if desired.
if train_adata.shape[0] > 0:
    train_adata = train_adata[np.random.RandomState(rng_seed).permutation(train_adata.shape[0])]
if validation_adata.shape[0] > 0:
    validation_adata = validation_adata[np.random.RandomState(rng_seed).permutation(validation_adata.shape[0])]
if test_adata.shape[0] > 0:
    test_adata = test_adata[np.random.RandomState(rng_seed).permutation(test_adata.shape[0])]

# 7. Print dataset statistics
print("\n--- Training Set ---")
print(f"Shape: {train_adata.shape}")
if train_adata.shape[0] > 0:
    print(f"Unique samples: {len(train_adata.obs['sample'].unique())}")
    print(f"MS samples: {np.sum(train_adata.obs['condition'] == 'MS')}")
    print(f"CTRL samples: {np.sum(train_adata.obs['condition'] == 'CTRL')}")
else:
    print("Training set is empty.")

print("\n--- Validation Set ---")
print(f"Shape: {validation_adata.shape}")
if validation_adata.shape[0] > 0:
    print(f"Unique samples: {len(validation_adata.obs['sample'].unique())}")
    print(f"MS samples: {np.sum(validation_adata.obs['condition'] == 'MS')}")
    print(f"CTRL samples: {np.sum(validation_adata.obs['condition'] == 'CTRL')}")
else:
    print("Validation set is empty.")

print("\n--- Test Set ---")
print(f"Shape: {test_adata.shape}")
if test_adata.shape[0] > 0:
    print(f"Unique samples: {len(test_adata.obs['sample'].unique())}")
    print(f"MS samples: {np.sum(test_adata.obs['condition'] == 'MS')}")
    print(f"CTRL samples: {np.sum(test_adata.obs['condition'] == 'CTRL')}")
else:
    print("Test set is empty.")

# 8. Extract features (X) and target (y) for each set
def extract_xy(adata, condition_label="MS"):
    if adata.shape[0] == 0:
        # Return empty DataFrame/Series with expected dtypes if set is empty
        return pd.DataFrame(columns=dataset.var_names), pd.Series(dtype='int')

    y = (adata.obs['condition'] == condition_label).astype(int)
    # Handle sparse matrices for X
    if hasattr(adata.X, "toarray"): # Checks if it's a sparse matrix
        x_data = adata.X.toarray()
    else:
        x_data = adata.X
    x = pd.DataFrame(x_data, columns=adata.var_names, index=adata.obs_names)
    return x, y

x_train, y_train = extract_xy(train_adata)
x_val, y_val = extract_xy(validation_adata)
x_test, y_test = extract_xy(test_adata)

print("\nExtracted X/y for train, validation, and test sets.")
print(f"x_train shape: {x_train.shape}, y_train shape: {y_train.shape}")
print(f"x_val shape: {x_val.shape}, y_val shape: {y_val.shape}")
print(f"x_test shape: {x_test.shape}, y_test shape: {y_test.shape}")

# Sanity check: ensure no sample ID overlap between sets
train_samples_set = set(train_sample_ids)
val_samples_set = set(validation_sample_ids)
test_samples_set = set(test_sample_ids)

if not train_samples_set.isdisjoint(val_samples_set):
    print(f"ERROR: Overlap between TRAIN and VALIDATION samples: {train_samples_set.intersection(val_samples_set)}")
if not train_samples_set.isdisjoint(test_samples_set):
    print(f"ERROR: Overlap between TRAIN and TEST samples: {train_samples_set.intersection(test_samples_set)}")
if not val_samples_set.isdisjoint(test_samples_set):
    print(f"ERROR: Overlap between VALIDATION and TEST samples: {val_samples_set.intersection(test_samples_set)}")

print("\nSplit complete.")


--- Sample Allocation (based on seed: 46) ---
Total unique samples: 15
Targeting 60% for Training: 9 samples
Targeting 20% for Validation: 3 samples
Targeting 20% (or remainder) for Test: 3 samples

Actual samples selected for Training: 9
Actual samples selected for Validation: 3
Actual samples selected for Test: 3

Selected samples for TRAINING set: ['71658', '95809', 'JSB_PBMC', '49131', '74594']... (first 5)
Selected samples for VALIDATION set: ['85037', 'YYW_PBMC', '83775']... (first 5)
Selected samples for TEST set: ['41540', '19270', 'KJS_PBMC']... (first 5)

--- Training Set ---
Shape: (1516, 17410)
Unique samples: 9
MS samples: 1077
CTRL samples: 439

--- Validation Set ---
Shape: (566, 17410)
Unique samples: 3
MS samples: 168
CTRL samples: 398

--- Test Set ---
Shape: (1423, 17410)
Unique samples: 3
MS samples: 876
CTRL samples: 547

Extracted X/y for train, validation, and test sets.
x_train shape: (1516, 17410), y_train shape: (1516,)
x_val shape: (566, 17410), y_val shape: 

In [6]:
from imblearn.combine import SMOTETomek
def apply_smotetomek(x_train: pd.DataFrame, y_train: pd.Series):
    """
    Applies SMOTETomek resampling to the training data.

    Args:
        x_train (pd.DataFrame): Training features.
        y_train (pd.Series): Training labels.

    Returns:
        tuple: Resampled training features and labels.
    """
    logging.info("Starting SMOTETomek resampling.")
    smote_tomek = SMOTETomek(random_state=42)
    logging.info("SMOTETomek object initialized.")
    x_resampled, y_resampled = smote_tomek.fit_resample(x_train, y_train)
    logging.info("SMOTETomek fit and resample complete.")
    logging.info(
        f"Shape of X_resampled: {x_resampled.shape}, shape of y_resampled:"
        f" {y_resampled.shape}"
    )
    return x_resampled, y_resampled

x_resampled, y_resampled = apply_smotetomek(x_train, y_train)

2025-06-15 15:22:47,867 - root - INFO - Starting SMOTETomek resampling.
2025-06-15 15:22:47,868 - root - INFO - SMOTETomek object initialized.
2025-06-15 15:22:54,948 - root - INFO - SMOTETomek fit and resample complete.
2025-06-15 15:22:54,949 - root - INFO - Shape of X_resampled: (2118, 17410), shape of y_resampled: (2118,)


In [7]:
import pandas as pd
from skopt import BayesSearchCV
from sklearn.model_selection import PredefinedSplit


def bayesianOpt(
    pipeline,
    hyperparameters,
    iteration,
    scorer,
    njobs,
    x_train,
    y_train,
    x_val,
    y_val,
):
    """
    Performs Bayesian optimization using a predefined validation set.

    Args:
        pipeline: The scikit-learn pipeline to optimize.
        hyperparameters: The hyperparameter search space for BayesSearchCV.
        iteration: The number of optimization iterations.
        scorer: The scoring metric to use.
        njobs: The number of parallel jobs to run.
        x_train: The training data features.
        y_train: The training data labels.
        x_val: The validation data features.
        y_val: The validation data labels.

    Returns:
        The BayesSearchCV result object.
    """

    # Create a predefined split using x_train and x_val
    test_fold = [-1] * len(x_train) + [0] * len(x_val)
    X = pd.concat([x_train, x_val], axis=0)
    y = pd.concat([y_train, y_val], axis=0)
    predefined_split = PredefinedSplit(test_fold)

    bayesianSearchResult = BayesSearchCV(
        estimator=pipeline,
        search_spaces=hyperparameters,
        cv=predefined_split,
        n_iter=iteration,
        return_train_score=True,
        refit="f1",
        scoring=scorer,
        n_jobs=njobs,
        verbose=3,
    )
    bayesianSearchResult.fit(X, y)
    print("Iperparametri:", bayesianSearchResult.best_params_)
    return bayesianSearchResult


In [8]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
import matplotlib.pyplot as plt
import numpy as np
import itertools
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline


def plot_confusion_matrix(
    cm, classes, normalize=False, title="Confusion matrix", cmap=plt.cm.Blues
):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    Dynamically adjusts text color for readability.
    """
    if normalize:
        cm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print("Confusion matrix, without normalization")

    print(cm)

    plt.imshow(cm, interpolation="nearest", cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = ".2f" if normalize else "d"
    thresh = cm.max() / 2.0
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        # Dynamically determine text color
        color = "white" if cm[i, j] < thresh else "black"  # Invert condition

        plt.text(
            j,
            i,
            format(cm[i, j], fmt),
            horizontalalignment="center",
            color=color,
        )

    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()


def prettyPrint(model, name, x_train, y_train, x_test, y_test, x_val, y_val, test=False):
    """
    Prints a formatted summary of a trained model's performance,
    including confusion matrix and ROC AUC on the *test* set when test=True.

    Args:
        model: The trained model (e.g., BayesSearchCV result).
        name: A descriptive name for the model.
        x_train: Training data features.
        y_train: Training data labels.
        x_test: Test data features.
        y_test: Test data labels.
        x_val: Validation data features.  (Not used in this version)
        y_val: Validation data labels. (Not used in this version)
        test: Whether to print test set results (including confusion matrix and ROC AUC).
    """

    print(name + ":")

    # Access best_params_ and best_score_ differently based on model type
    if isinstance(model, (GridSearchCV, RandomizedSearchCV)):
        print("Iperparametri: ", model.best_params_)
        print("Mean f1 cross-validated: ", model.best_score_)
        best_index = model.best_index_

        # Extract metrics names safely
        precision_0_key = "mean_test_precision 0"
        recall_0_key = "mean_test_recall 0"
        precision_1_key = "mean_test_precision 1"
        recall_1_key = "mean_test_recall 1"
        accuracy_key = "mean_test_Accuracy"
        f1_key = "mean_test_f1"

        # Function to safely get the values, returns NaN if key not found
        def get_metric(key, index):
            return model.cv_results_.get(
                key, [float("NaN")] * len(model.cv_results_["params"])
            )[index]

        # Extract the metrics
        precision_0 = get_metric(precision_0_key, best_index)
        recall_0 = get_metric(recall_0_key, best_index)
        precision_1 = get_metric(precision_1_key, best_index)
        recall_1 = get_metric(recall_1_key, best_index)
        accuracy = get_metric(accuracy_key, best_index)
        f1 = get_metric(f1_key, best_index)

        print("Train f1: ", model.score(x_train, y_train))
        print("\t\t precision \t\t recall \t\t f1-score")
        print(f"0 \t\t {precision_0:.2f} \t\t\t {recall_0:.2f}")
        print(f"1 \t\t {precision_1:.2f} \t\t\t {recall_1:.2f}")
        print(f"Accuracy \t\t\t\t\t\t\t {accuracy:.2f}")

        macro_avg_precision = (precision_0 + precision_1) / 2
        macro_avg_recall = (recall_0 + recall_1) / 2

        print(
            f"macro avg \t {macro_avg_precision:.2f} \t\t\t {macro_avg_recall:.2f} \t\t\t {f1:.2f}"
        )

    else:  # For Pipeline or other models without best_params_
        print("Iperparametri: ", model.get_params())
        print("Train f1: ", model.score(x_train, y_train))

    if test:
        print("\nTest Set Evaluation:")
        y_test_pred = model.predict(x_test)

        # Confusion Matrix
        cnf_matrix = confusion_matrix(y_test, y_test_pred)
        plt.figure()
        plot_confusion_matrix(
            cnf_matrix, classes=["0", "1"], title="Confusion Matrix - Test Set"
        )
        plt.show()

        # ROC AUC
        try:  # Some classifiers might not have predict_proba
            y_test_proba = model.predict_proba(x_test)[:, 1]
            roc_auc = roc_auc_score(y_test, y_test_proba)
            fpr, tpr, _ = roc_curve(y_test, y_test_proba)

            plt.figure()
            plt.plot(fpr, tpr, label=f"ROC curve (area = {roc_auc:.2f})")
            plt.plot([0, 1], [0, 1], "k--")
            plt.xlim([0.0, 1.0])
            plt.ylim([0.0, 1.05])
            plt.xlabel("False Positive Rate")
            plt.ylabel("True Positive Rate")
            plt.title("Receiver Operating Characteristic - Test Set")
            plt.legend(loc="lower right")
            plt.show()

            print(f"ROC AUC score: {roc_auc:.2f}")
        except AttributeError:
            print("Model does not support probability predictions for ROC AUC calculation.")

        print("\nTest f1 score: ", model.score(x_test, y_test))
        print(classification_report(y_test, y_test_pred), "\n\n")


# Example Usage (replace with your actual data)
# Assuming you have x_train, y_train, x_test, y_test, x_val, y_val loaded
# MODEL TEST
# model = joblib.load("bayesianOptResult.pkl")
# prettyPrint(model, "CD4 PBMC", x_train, y_train, x_test, y_test, x_val, y_val, True)


In [9]:
pipeline = Pipeline(steps=[('scaling', MinMaxScaler()), ('classifier', XGBClassifier(random_state=42, device="cuda"))])

def precision_class_0(y_true, y_pred):
    return precision_score(y_true, y_pred, average=None)[0]

def precision_class_1(y_true, y_pred):
    return precision_score(y_true, y_pred, average=None)[1]

def recall_class_0(y_true, y_pred):
    return recall_score(y_true, y_pred, average=None)[0]

def recall_class_1(y_true, y_pred):
    return recall_score(y_true, y_pred, average=None)[1]

scorer = {
    'Accuracy': 'accuracy',
    'precision 0': make_scorer(precision_class_0),
    'precision 1': make_scorer(precision_class_1),
    'recall 0': make_scorer(recall_class_0),
    'recall 1': make_scorer(recall_class_1),
    'f1': make_scorer(f1_score, average='macro')
}

In [19]:
param_dist = {
    'classifier__n_estimators': Integer(50, 3000),  # Numero di alberi
    'classifier__max_depth': Integer(2, 15),  # Profondità dell'albero
    'classifier__learning_rate': Real(0.0000001, 1, prior='log-uniform'),  # Tasso di apprendimento
    'classifier__gamma': Real(0.000001, 1, prior='log-uniform'),  # Minimum loss reduction
    'classifier__min_child_weight': Integer(1, 8), 
    'classifier__scale_pos_weight': Categorical([1, 439/1077, 4]),
    'classifier__reg_alpha': Real(0.000001, 100, prior='log-uniform'),  # Regolarizzazione L1
    'classifier__reg_lambda': Real(0.000001, 100, prior='log-uniform')  # Regolarizzazione L2
}
bayesianOptResult = bayesianOpt(pipeline, param_dist, 500, scorer, 1, x_resampled, y_resampled, x_val, y_val)
joblib.dump(bayesianOptResult, "bayesianOptResult.pkl")

Fitting 1 folds for each of 1 candidates, totalling 1 fits


/usr/local/lib/python3.10/dist-packages/xgboost/core.py:160: UserWarning: [16:37:19] WARNING: /home/coder/xgboost/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is 

[CV 1/1] END classifier__gamma=0.030557508018308144, classifier__learning_rate=8.39706648415488e-06, classifier__max_depth=3, classifier__min_child_weight=6, classifier__n_estimators=2221, classifier__reg_alpha=10.833681922641679, classifier__reg_lambda=3.3931688755298133, classifier__scale_pos_weight=4; Accuracy: (train=0.500, test=0.297) f1: (train=0.333, test=0.229) precision 0: (train=0.000, test=0.000) precision 1: (train=0.500, test=0.297) recall 0: (train=0.000, test=0.000) recall 1: (train=1.000, test=1.000) total time=  17.3s
Fitting 1 folds for each of 1 candidates, totalling 1 fits
[CV 1/1] END classifier__gamma=1.8559547208442095e-06, classifier__learning_rate=4.899526000215692e-05, classifier__max_depth=3, classifier__min_child_weight=4, classifier__n_estimators=2392, classifier__reg_alpha=0.014334649338816459, classifier__reg_lambda=4.349399534360058e-06, classifier__scale_pos_weight=1; Accuracy: (train=0.742, test=0.611) f1: (train=0.738, test=0.553) precision 0: (train=

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/m

[CV 1/1] END classifier__gamma=0.07589193399426075, classifier__learning_rate=1.4062228730601417e-07, classifier__max_depth=6, classifier__min_child_weight=7, classifier__n_estimators=2640, classifier__reg_alpha=3.9148964299911486, classifier__reg_lambda=15.862762436747731, classifier__scale_pos_weight=0.4076137418755803; Accuracy: (train=0.500, test=0.703) f1: (train=0.333, test=0.413) precision 0: (train=0.500, test=0.703) precision 1: (train=0.000, test=0.000) recall 0: (train=1.000, test=1.000) recall 1: (train=0.000, test=0.000) total time=  41.2s
Fitting 1 folds for each of 1 candidates, totalling 1 fits


...

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/m

[CV 1/1] END classifier__gamma=1e-06, classifier__learning_rate=1.0, classifier__max_depth=15, classifier__min_child_weight=8, classifier__n_estimators=3000, classifier__reg_alpha=100.0, classifier__reg_lambda=1e-06, classifier__scale_pos_weight=0.4076137418755803; Accuracy: (train=0.500, test=0.703) f1: (train=0.333, test=0.413) precision 0: (train=0.500, test=0.703) precision 1: (train=0.000, test=0.000) recall 0: (train=1.000, test=1.000) recall 1: (train=0.000, test=0.000) total time=  13.1s
Fitting 1 folds for each of 1 candidates, totalling 1 fits
[CV 1/1] END classifier__gamma=1.0, classifier__learning_rate=3.677796750817893e-06, classifier__max_depth=2, classifier__min_child_weight=8, classifier__n_estimators=3000, classifier__reg_alpha=0.14370503436483967, classifier__reg_lambda=1e-06, classifier__scale_pos_weight=1; Accuracy: (train=0.662, test=0.648) f1: (train=0.644, test=0.542) precision 0: (train=0.611, test=0.726) precision 1: (train=0.801, test=0.376) recall 0: (train=0

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/m

[CV 1/1] END classifier__gamma=1e-06, classifier__learning_rate=1e-07, classifier__max_depth=15, classifier__min_child_weight=8, classifier__n_estimators=50, classifier__reg_alpha=100.0, classifier__reg_lambda=33.004298562660885, classifier__scale_pos_weight=0.4076137418755803; Accuracy: (train=0.500, test=0.703) f1: (train=0.333, test=0.413) precision 0: (train=0.500, test=0.703) precision 1: (train=0.000, test=0.000) recall 0: (train=1.000, test=1.000) recall 1: (train=0.000, test=0.000) total time=   5.2s
Fitting 1 folds for each of 1 candidates, totalling 1 fits
[CV 1/1] END classifier__gamma=3.528640993891334e-05, classifier__learning_rate=0.04572524204098439, classifier__max_depth=4, classifier__min_child_weight=8, classifier__n_estimators=1559, classifier__reg_alpha=100.0, classifier__reg_lambda=1e-06, classifier__scale_pos_weight=1; Accuracy: (train=0.702, test=0.698) f1: (train=0.697, test=0.622) precision 0: (train=0.662, test=0.770) precision 1: (train=0.766, test=0.490) rec

['bayesianOptResult.pkl']

In [11]:
# Refittiamo il modello su tutto il dataset
model = joblib.load("bayesianOptResult.pkl")
prettyPrint(model, "bcells pbmc", x_train, y_train, x_test, y_test, x_val, y_val, test=False)

bcells pbmc:
Iperparametri:  {'cv': PredefinedSplit(test_fold=array([-1, -1, ...,  0,  0])), 'error_score': 'raise', 'estimator__memory': None, 'estimator__steps': [('scaling', MinMaxScaler()), ('classifier', XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device='cuda', early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, random_state=42, ...))], 'estimator__verbose': False, '

/usr/local/lib/python3.10/dist-packages/xgboost/core.py:160: UserWarning: [15:23:23] WARNING: /home/coder/xgboost/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


In [23]:
import xgboost as xgb
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline

# Define the best parameters found by BayesSearchCV
best_params = {
    "gamma": 0.00010424606377892094,
    "learning_rate": 0.08395998719982567,
    "max_depth":3,
    "min_child_weight": 8,
    "n_estimators": 1095,
    "reg_alpha": 100.0,
    "reg_lambda": 100.0,
    "scale_pos_weight":1,
}

# Create XGBoost classifier with the best parameters
xgb_classifier = xgb.XGBClassifier(
    objective='binary:logistic',
    use_label_encoder=False,
    eval_metric='logloss',
    seed=42,
    tree_method='hist',  # Add tree_method for GPU support
    device="cuda",
    **best_params  # Pass best_params directly
)

# Create a pipeline with MinMaxScaler and XGBoost
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),  # MinMaxScaler
    ('classifier', xgb_classifier)  # XGBoost classifier
])

# Fit the pipeline on the resampled training data
pipeline.fit(x_resampled, y_resampled)

print("Fitted pipeline on the resampled training data.")

# Make predictions on the validation set
y_pred_val = pipeline.predict(x_val)

# Evaluate predictions on the validation set
accuracy = accuracy_score(y_val, y_pred_val)
precision = precision_score(y_val, y_pred_val)
recall = recall_score(y_val, y_pred_val)
f1 = f1_score(y_val, y_pred_val)
roc_auc = roc_auc_score(y_val, pipeline.predict_proba(x_val)[:, 1])
confusion = confusion_matrix(y_val, y_pred_val)

print("\n--- Evaluation Metrics on Validation Set ---")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")
print("\nConfusion Matrix:")
print(confusion)


Fitted pipeline on the resampled training data.

--- Evaluation Metrics on Validation Set ---
Accuracy: 0.7155
Precision: 0.5238
Recall: 0.4583
F1-Score: 0.4889
ROC AUC: 0.6412

Confusion Matrix:
[[328  70]
 [ 91  77]]


In [24]:
import xgboost as xgb
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline

# Define the best parameters found by BayesSearchCV
best_params = {
    "gamma": 0.00010424606377892094,
    "learning_rate": 0.08395998719982567,
    "max_depth":3,
    "min_child_weight": 8,
    "n_estimators": 1095,
    "reg_alpha": 100.0,
    "reg_lambda": 100.0,
    "scale_pos_weight":1,
}

# Create XGBoost classifier with the best parameters
xgb_classifier = xgb.XGBClassifier(
    objective='binary:logistic',
    use_label_encoder=False,
    eval_metric='logloss',
    seed=42,
    tree_method='hist',  # Add tree_method for GPU support
    device="cuda",
    **best_params  # Pass best_params directly
)

# Create a pipeline with MinMaxScaler and XGBoost
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),  # MinMaxScaler
    ('classifier', xgb_classifier)  # XGBoost classifier
])

# Combine the training and validation data
X_full = pd.concat([x_train, x_val], axis=0)
y_full = pd.concat([y_train, y_val], axis=0)

# Fit the pipeline on the full dataset
pipeline.fit(X_full, y_full)

print("Fitted pipeline on the combined training and validation data.")

# Make predictions on the test set
y_pred_test = pipeline.predict(x_test)

# Evaluate predictions on the test set
accuracy = accuracy_score(y_test, y_pred_test)
precision = precision_score(y_test, y_pred_test)
recall = recall_score(y_test, y_pred_test)
f1 = f1_score(y_test, y_pred_test)
roc_auc = roc_auc_score(y_test, pipeline.predict_proba(x_test)[:, 1])
confusion = confusion_matrix(y_test, y_pred_test)

print("\n--- Evaluation Metrics on Test Set ---")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")
print("\nConfusion Matrix:")
print(confusion)


Fitted pipeline on the combined training and validation data.

--- Evaluation Metrics on Test Set ---
Accuracy: 0.7765
Precision: 0.7642
Recall: 0.9212
F1-Score: 0.8354
ROC AUC: 0.7330

Confusion Matrix:
[[298 249]
 [ 69 807]]


In [25]:
import xgboost as xgb
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline

# Define the best parameters found by BayesSearchCV
best_params = {
    "gamma": 0.00010424606377892094,
    "learning_rate": 0.08395998719982567,
    "max_depth":3,
    "min_child_weight": 8,
    "n_estimators": 1095,
    "reg_alpha": 100.0,
    "reg_lambda": 100.0,
    "scale_pos_weight":1,
}
# Create XGBoost classifier with the best parameters
xgb_classifier = xgb.XGBClassifier(
    objective='binary:logistic',
    use_label_encoder=False,
    eval_metric='logloss',
    seed=42,
    tree_method='hist',  # Add tree_method for GPU support
    device="cuda",
    **best_params  # Pass best_params directly
)

# Create a pipeline with MinMaxScaler and XGBoost
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),  # MinMaxScaler
    ('classifier', xgb_classifier)  # XGBoost classifier
])

# Combine the training and validation data
X_full = pd.concat([x_train, x_val, x_test], axis=0)
y_full = pd.concat([y_train, y_val, y_test], axis=0)

# Fit the pipeline on the full dataset
pipeline.fit(X_full, y_full)

print("Fitted pipeline on the combined training, validation data and test data.")

#inutile fare predizioni qui

#salviamo
joblib.dump(pipeline, "BestModelRefittedWholeDataset.pkl")


Fitted pipeline on the combined training, validation data and test data.


['BestModelRefittedWholeDataset.pkl']

In [ ]:
print(train_sample_ids)
print(validation_sample_ids)
print(test_sample_ids)